<a href="https://colab.research.google.com/github/CharalapML/ColabCode/blob/main/Cld_FIXED_Aug2026_WORKS_Ringing_Up_VIDEO_v6_Transparent_Bell_Swinging_Dual_canvas.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Claude — Ringing Up / Bell Swinging (fixed)

Based on **Aug2026 TEST for WORKS_Ringing_Up-VIDEO_v6 Transparent_Bell_Swinging_Dual canvas_ipynb**.

### What was wrong

**The jump at the end of the mp4 was caused by stale frames, not by the angle list.**
`generate_frames()` called `os.makedirs(FRAMES_DIR, exist_ok=True)` but never emptied
the folder, and `create_video()` collected frames with
`sorted(glob.glob("frames/frame_*.png"))`. Any run that produced *more* frames than the
current one left higher-numbered PNGs on disk; they sort last, so they were appended to
the end of the video. Result: the animation plays, jumps to unrelated content, then
"settles" as it plays out the old tail.

### Fixes applied

| # | Fix |
|---|-----|
| 1 | `frames/` is deleted and recreated on every run (`shutil.rmtree`). |
| 2 | `create_video()` builds its file list from `frames_deg`, not from `glob` — leftovers can never be picked up. |
| 3 | Frame count and frame size are asserted before encoding; `VideoWriter` silently drops mismatched frames otherwise. |
| 4 | **One** angle list (`ANGLES_DEG` in the config cell). Previously the list was redefined in the render cell but the render loop read `frames_deg` from an earlier cell, so edits there did nothing. |
| 5 | The download cell is now **last**. Previously it ran before the render, so it downloaded whatever `video_v6.mp4` was left over from an earlier run. |
| 6 | Sweeps land exactly on each endpoint with evenly-sized steps (`range()` overshot, leaving 1-2 deg final steps). |
| 7 | Every endpoint is held for exactly `PAUSE_FRAMES` frames. Before, `range()` excluded the endpoint so interior endpoints were held 3 frames while `-186` was written 4x (3 pause frames plus a trailing `append`). |
| 8 | Removed unused `TOTAL_FRAMES = 600`, `object_spin_angle`, `object_spin_angle_rad`. |

### Unchanged on purpose

Scale (0.7), background scale (0.7), orbit centre (745, 800), pivot (`w/2`, `h/2`) and the
angle list all keep their original values, so output should match the intended animation.
See the note on `PIVOT_MODE` in the config cell — the original code contained two
contradictory pivot calculations and the centre one won.


In [ ]:
from google.colab import drive
drive.mount('/content/gdrive')

Mounted at /content/gdrive


In [ ]:
!pip install cairosvg

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 46.0/46.0 kB 3.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 75.6/75.6 kB 6.0 MB/s eta 0:00:00


In [ ]:
import os
import glob
import math
import shutil

import cv2
import cairosvg
from xml.etree import ElementTree as ET

## Configuration — the single source of truth

Every knob lives here. Nothing below redefines these.

In [ ]:
# ---------------------------------------------------------------- paths
GDRIVE      = "/content/gdrive/My Drive/Images_png_svg_jpg_etc"
BG_SVG      = f"{GDRIVE}/FrameOnly.svg"
OBJ_SVG     = f"{GDRIVE}/BellinWheel.svg"

SCENE_SVG   = "scene.svg"
FRAMES_DIR  = "frames"
VIDEO_NAME  = "video_v6.mp4"

SVG_NS      = "http://www.w3.org/2000/svg"

# ------------------------------------------------------------- animation
# FIX 4: this is the ONLY place the angle list is defined.
ANGLES_DEG   = [0, 31, -31, 62, -62, 93, -93, 124, -124, 155, -155, 186, -186]

STEP_DEG     = 3      # max degrees travelled per frame
FPS          = 30
PAUSE_SEC    = 0.1    # hold at each end of the swing
PAUSE_FRAMES = int(PAUSE_SEC * FPS)

# ---------------------------------------------------------------- layout
BG_SCALE     = 0.7    # background
OBJ_SCALE    = 0.7    # bell
ORBIT_X      = 745
ORBIT_Y      = 800
ORBIT_RADIUS = 0

# PIVOT_MODE — the original notebook computed this twice, contradictorily:
#   one cell used viewBox top-centre  (w/2, 0)   <- what the comment said it wanted
#   a later cell overwrote it with    (w/2, h/2) <- what actually ran
# "centre" reproduces the original output. Switch to "top" if the bell should
# hang from its mount rather than rotate about its middle.
PIVOT_MODE   = "centre"     # "centre" | "top"

CLEAN_FRAMES = True   # FIX 1: wipe frames/ before each run. Leave True.

## Build the scene

In [ ]:
ET.register_namespace("", SVG_NS)

bg_root  = ET.parse(BG_SVG).getroot()
obj_root = ET.parse(OBJ_SVG).getroot()

scene_root = ET.Element(bg_root.tag, bg_root.attrib)

bg_group = ET.SubElement(scene_root, f"{{{SVG_NS}}}g")
bg_group.set("id", "background")
bg_group.set("transform", f"scale({BG_SCALE})")
bg_group.extend(list(bg_root))

obj_group = ET.SubElement(scene_root, f"{{{SVG_NS}}}g")
obj_group.set("id", "movable_object")
obj_group.extend(list(obj_root))

ET.ElementTree(scene_root).write(SCENE_SVG, encoding="utf-8", xml_declaration=True)
print("wrote", SCENE_SVG)

wrote scene.svg


## Pivot

One calculation, not two.

In [ ]:
width_attr  = obj_root.get("width")
height_attr = obj_root.get("height")

if width_attr and height_attr:
    bell_width  = float(width_attr.replace("px", ""))
    bell_height = float(height_attr.replace("px", ""))
else:
    view_box = obj_root.get("viewBox")
    if view_box:
        _, _, bell_width, bell_height = map(float, view_box.split())
    else:
        print("WARNING: no width/height and no viewBox on the object SVG; falling back to 100x100.")
        bell_width = bell_height = 100.0

bell_pivot_local_x = bell_width / 2
bell_pivot_local_y = bell_height / 2 if PIVOT_MODE == "centre" else 0.0

print(f"bell        = {bell_width} x {bell_height}")
print(f"pivot       = ({bell_pivot_local_x}, {bell_pivot_local_y})   [PIVOT_MODE={PIVOT_MODE}]")

bell        = 1600.0 x 1613.0
pivot       = (800.0, 806.5)   [PIVOT_MODE=centre]


## Frame sequence

`sweep()` divides each leg into equal steps of **at most** `STEP_DEG` and lands exactly on
the endpoint. The old `range(start, end, step)` overshot, so the final step of each leg was
1-2 deg instead of 3 — a small stutter at every turnaround.

In [ ]:
def sweep(start, end, step):
    """Angles from start to end, inclusive of end, exclusive of start.

    Steps are equal in size and never exceed `step` degrees.
    """
    if start == end:
        return []
    n = math.ceil(abs(end - start) / step)
    return [start + (end - start) * k / n for k in range(1, n + 1)]


def build_frames(angles, step, pause_frames):
    frames = [float(angles[0])]
    for start, end in zip(angles[:-1], angles[1:]):
        legs = sweep(start, end, step)
        frames.extend(legs)
        # FIX 7: sweep() already lands ON the endpoint, so that arrival frame counts
        # as the first frame of the hold. Adding a full pause_frames here would hold
        # every endpoint for pause_frames + 1.
        frames.extend([float(end)] * (pause_frames - 1))
    return frames


frames_deg = build_frames(ANGLES_DEG, STEP_DEG, PAUSE_FRAMES)

print(f"frames   : {len(frames_deg)}")
print(f"duration : {len(frames_deg) / FPS:.2f} s @ {FPS} fps")
print(f"first 6  : {[round(a, 1) for a in frames_deg[:6]]}")
print(f"last 6   : {[round(a, 1) for a in frames_deg[-6:]]}")

steps = [abs(b - a) for a, b in zip(frames_deg, frames_deg[1:]) if b != a]
print(f"max step : {max(steps):.2f} deg  (limit {STEP_DEG})")

frames   : 835
duration : 27.83 s @ 30 fps
first 6  : [0.0, 2.8, 5.6, 8.5, 11.3, 14.1]
last 6   : [-177.0, -180.0, -183.0, -186.0, -186.0, -186.0]
max step : 3.00 deg  (limit 3)


## Render and encode

In [ ]:
def update_object_transform(tree, final_x, final_y, scale_factor, spin_angle,
                            pivot_x, pivot_y):
    root = tree.getroot()

    # SVG applies transforms right-to-left:
    #   4. move the pivot to its place in the scene
    #   3. scale about (0,0)
    #   2. rotate about (0,0)
    #   1. move the object's local pivot to (0,0)
    transform_string = (
        f"translate({final_x} {final_y}) "
        f"scale({scale_factor}) "
        f"rotate({spin_angle}) "
        f"translate({-pivot_x} {-pivot_y})"
    )

    for el in root.iter(f"{{{SVG_NS}}}g"):
        if el.get("id") == "movable_object":
            el.set("transform", transform_string)
    return tree


def frame_path(i, ext):
    return os.path.join(FRAMES_DIR, f"frame_{i:06d}.{ext}")


def generate_frames():
    # FIX 1: never inherit frames from a previous run.
    if CLEAN_FRAMES and os.path.isdir(FRAMES_DIR):
        shutil.rmtree(FRAMES_DIR)
    os.makedirs(FRAMES_DIR, exist_ok=True)

    final_x = ORBIT_X + ORBIT_RADIUS
    final_y = ORBIT_Y + ORBIT_RADIUS

    for i, spin_angle in enumerate(frames_deg):
        tree = ET.parse(SCENE_SVG)
        tree = update_object_transform(
            tree, final_x, final_y, OBJ_SCALE, spin_angle,
            bell_pivot_local_x, bell_pivot_local_y,
        )

        svg_path = frame_path(i, "svg")
        tree.write(svg_path, encoding="utf-8", xml_declaration=True)
        cairosvg.svg2png(url=svg_path, write_to=frame_path(i, "png"),
                         background_color="white")

        if (i + 1) % 100 == 0 or i + 1 == len(frames_deg):
            print(f"  rendered {i + 1}/{len(frames_deg)}")


def create_video():
    # FIX 2: the file list comes from frames_deg, never from glob().
    images = [frame_path(i, "png") for i in range(len(frames_deg))]

    missing = [p for p in images if not os.path.exists(p)]
    if missing:
        raise FileNotFoundError(f"{len(missing)} frame(s) missing, first: {missing[0]}")

    # FIX 2 (cont.): prove nothing extra is on disk.
    on_disk = glob.glob(os.path.join(FRAMES_DIR, "frame_*.png"))
    if len(on_disk) != len(images):
        raise RuntimeError(
            f"{len(on_disk)} PNGs on disk but {len(images)} expected — "
            "stale frames present. Set CLEAN_FRAMES = True and re-run."
        )

    first = cv2.imread(images[0])
    if first is None:
        raise RuntimeError(f"could not read {images[0]}")
    height, width, _ = first.shape

    fourcc = cv2.VideoWriter_fourcc(*"mp4v")
    writer = cv2.VideoWriter(VIDEO_NAME, fourcc, FPS, (width, height))

    written = 0
    for path in images:
        frame = cv2.imread(path)
        if frame is None:
            raise RuntimeError(f"could not read {path}")
        # FIX 3: VideoWriter silently discards wrong-sized frames.
        if frame.shape[:2] != (height, width):
            raise ValueError(
                f"{path} is {frame.shape[1]}x{frame.shape[0]}, expected {width}x{height}"
            )
        writer.write(frame)
        written += 1

    writer.release()

    if written != len(frames_deg):
        raise RuntimeError(f"wrote {written} frames, expected {len(frames_deg)}")

    print(f"{VIDEO_NAME}: {written} frames, {width}x{height}, "
          f"{written / FPS:.2f}s @ {FPS}fps")


generate_frames()
create_video()

  rendered 100/835
  rendered 200/835
  rendered 300/835
  rendered 400/835
  rendered 500/835
  rendered 600/835
  rendered 700/835
  rendered 800/835
  rendered 835/835
video_v6.mp4: 835 frames, 2090x2352, 27.83s @ 30fps


## Verify

Confirms the encoded file matches the sequence. A mismatch here is what produced the
jump at the end of the old videos.

In [ ]:
cap = cv2.VideoCapture(VIDEO_NAME)
actual = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
fps    = cap.get(cv2.CAP_PROP_FPS)
w      = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
h      = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
cap.release()

print(f"expected frames : {len(frames_deg)}")
print(f"actual frames   : {actual}")
print(f"fps / size      : {fps} / {w}x{h}")
print(f"size on disk    : {os.path.getsize(VIDEO_NAME):,} bytes")

if actual == len(frames_deg):
    print("\nOK — no stale frames.")
else:
    print(f"\nMISMATCH: {actual - len(frames_deg):+d} frames. Investigate before downloading.")

expected frames : 835
actual frames   : 835
fps / size      : 30.0 / 2090x2352
size on disk    : 42,725,089 bytes

OK — no stale frames.


## Download

**This cell is last on purpose.** In the original notebook it ran before the render, so it
downloaded whatever `video_v6.mp4` happened to be left over from a previous run.

In [ ]:
from google.colab import files

if os.path.exists(VIDEO_NAME):
    files.download(VIDEO_NAME)
else:
    print(f"{VIDEO_NAME} not found — run the render cell first.")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>